# MRM-Trace starter: how long does data live in each memory region of an LLM?

This dataset is hardware-sampled memory accesses (`perf mem`, Intel PEBS) from llama.cpp inference on CPU,
each labelled by memory region: model **weights**, **KV cache** or **runtime**. This notebook loads the
tables, shows the headline comparison, and plots one run's trace.

Read the *Sampling* section of the dataset description first: accesses are sampled, stores 500x more
sparsely than loads, so some quantities are null by design rather than zero.

In [ ]:
import glob, os
import pandas as pd
import matplotlib.pyplot as plt

# Works on Kaggle (/kaggle/input/<slug>/) and locally (datasets/mrm-trace-kaggle/).
candidates = glob.glob("/kaggle/input/*/runs.csv") + glob.glob("datasets/mrm-trace-kaggle/runs.csv")
DATA = os.path.dirname(candidates[0])
runs = pd.read_csv(f"{DATA}/runs.csv")
metrics = pd.read_csv(f"{DATA}/metrics.csv")
runs[["run_id", "model_id", "model.params_billions", "output_tokens", "rep", "run_validity", "avg_tok_per_s"]].head()

## Headline: each region type across all runs

In [ ]:
pd.read_csv(f"{DATA}/summary_by_region.csv").set_index("region_type").T

## Retention by region and model

Retention is the time between a region's first and last sampled access. It is only defined for regions
sampled at least twice (`n_regions_observable`).

In [ ]:
m = metrics[metrics.region_type.isin(["weight", "kv_cache", "runtime"])]
piv = m.groupby(["model_id", "region_type"]).retention_p50_s.mean().unstack()
order = runs.drop_duplicates("model_id").sort_values("model.params_billions").model_id
ax = piv.loc[order].plot.bar(figsize=(8, 4), rot=0)
ax.set_ylabel("median retention (s)"); ax.set_xlabel("")
ax.set_title("Median region retention, mean over runs"); plt.tight_layout()

## KV-cache blocks: how many had their write observed?

A block's lifetime needs its write. Where no store was sampled, `lifetime_ns` is null, not 0.

In [ ]:
kv = pd.read_parquet(f"{DATA}/kv_block_lifecycle.parquet")
print(f"{len(kv):,} blocks; write observed for {kv.write_observed.mean():.1%}")
(kv.dropna(subset=["lifetime_ns"]).lifetime_ns / 1e9).describe()

## One run's trace, coloured by region

In [ ]:
run_id = runs.run_id.iloc[0]
t = pd.read_parquet(f"{DATA}/trace.parquet", filters=[("run_id", "==", run_id)],
                    columns=["timestamp_ns", "address_page", "region_type", "op_type", "cache_level", "sample_weight"])
print(f"{len(t):,} rows; weighted estimate of full trace: {t.sample_weight.sum():,.0f}")
t["t_s"] = (t.timestamp_ns - t.timestamp_ns.min()) / 1e9
t["page_rank"] = t.address_page.rank(method="dense")
fig, ax = plt.subplots(figsize=(10, 4))
for region, s, c in [("runtime", 0.3, "#1baf7a"), ("weight", 1, "#2a78d6"), ("kv_cache", 2, "#eb6834")]:
    g = t[t.region_type == region]
    g = g.sample(min(len(g), 80_000), random_state=0)
    ax.scatter(g.t_s, g.page_rank, s=s, color=c, label=region, linewidths=0)
ax.set_xlabel("seconds into run"); ax.set_ylabel("address (page order)"); ax.set_yticks([])
ax.legend(markerscale=8); ax.set_title(run_id); plt.tight_layout()

## Where accesses were served from, by region

In [ ]:
(t[t.op_type == "load"]
   .groupby(["region_type", "cache_level"]).sample_weight.sum()
   .groupby(level=0).transform(lambda s: s / s.sum())
   .unstack().round(3))